<a href="https://colab.research.google.com/github/afifamaisha13/Machine_learing-and-Deep_learning/blob/main/Feature_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import pandas as pd

In [2]:
df= sns.load_dataset('iris')

In [3]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


In [ ]:
df.shape

(150, 5)

In [ ]:
df.isnull().sum()

,0
sepal_length,0
sepal_width,0
petal_length,0
petal_width,0
species,0


In [4]:
X = df.drop('species', axis=1)
y = df['species']

In [5]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [7]:
from sklearn.ensemble import RandomForestClassifier

In [8]:
rf_model = RandomForestClassifier(
    n_estimators=100,     # number of trees
    max_depth=None,       # let trees grow fully
    random_state=42,

)

#RFE

In [ ]:
rfe= RFE(estimator=rf_model, n_features_to_select=2)

In [ ]:
rfe.fit(X_train, y_train)

RFE(estimator=RandomForestClassifier(random_state=42), n_features_to_select=2)

In [ ]:
print("Selected Features:")
print(X.columns[rfe.support_])

Selected Features:
Index(['petal_length', 'petal_width'], dtype='object')


#**A ranking of 1 means the feature was selected as one of the most important features. Higher numbers indicate less important features that were eliminated earlier in the recursive process.**

In [ ]:
#feature ranking
print("\nRanking:")
print(pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rfe.ranking_
}))


Ranking:
        Feature  Ranking
0  sepal_length        2
1   sepal_width        3
2  petal_length        1
3   petal_width        1


In [ ]:
#feature importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.fit(X_train, y_train).feature_importances_
})

importance = importance.sort_values(by='Importance', ascending=False)

print(importance)
# as output shows petal_width and petal_length has more importance than the two others so our ranking was correct

        Feature  Importance
3   petal_width    0.437185
2  petal_length    0.431466
0  sepal_length    0.116349
1   sepal_width    0.015000




#Train the dataset only using the selected two features using random forest model and compare with previous accuracy 0.9
(from ensemble learning)


In [ ]:
X_train_rfe = rfe.transform(X_train)
X_test_rfe = rfe.transform(X_test)


In [ ]:
# Train Random Forest on selected features
rf_selected = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [ ]:
rf_selected.fit(X_train_rfe, y_train)


RandomForestClassifier(random_state=42)

In [ ]:
y_pred = rf_selected.predict(X_test_rfe)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
accuracy

0.9666666666666667

# as we can see accuracy has improved from 0.9 to 0.97 after using RFE(feature selection technique)

#Lasso Technique (This is an excellent question. The reason is LASSO itself is not a machine learning classifier for classification problems. It is a regularization technique that must be applied to a model that has coefficients (weights).

For the Iris dataset, which is a classification problem, we use Logistic Regression with L1 regularization because L1 regularization is the implementation of LASSO for classification.

**Why Logistic Regression?**

Your Iris dataset contains three flower classes:

Setosa
Versicolor
Virginica

Since the target variable is categorical, this is a classification problem.

For classification, LASSO is implemented as:

LogisticRegression(
    penalty='l1'
)

The penalty='l1' tells Logistic Regression to apply LASSO regularization.

Why not use Lasso()?

Scikit-learn also has a model called:

from sklearn.linear_model import Lasso

However, this is only for regression, where the target is continuous.)

In [39]:
lasso = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=0.05,
    random_state=42
)

lasso.fit(X_train, y_train)

LogisticRegression(C=0.05, penalty='l1', random_state=42, solver='liblinear')

In [41]:
#Display LASSO Coefficients
coef = pd.DataFrame(
    lasso.coef_.T,
    index=X.columns,
    columns=['Class 0', 'Class 1', 'Class 2']
)

print("LASSO Coefficients:")
print(coef)

LASSO Coefficients:
               Class 0   Class 1   Class 2
sepal_length  0.000000  0.000000 -0.137055
sepal_width   0.540427 -0.191584 -0.316145
petal_length -0.766369  0.000000  0.340133
petal_width   0.000000  0.000000  0.000000


In [42]:
#Calculate Feature Importance
#We use the maximum absolute coefficient across the three classes.

importance = coef.abs().max(axis=1)

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importance
})

print("\nFeature Importance:")
print(feature_importance)


Feature Importance:
                   Feature  Importance
sepal_length  sepal_length    0.137055
sepal_width    sepal_width    0.540427
petal_length  petal_length    0.766369
petal_width    petal_width    0.000000


In [43]:
threshold = 0.5

selected_features = feature_importance[
    feature_importance['Importance'] >= threshold
]

print("\nSelected Features:")
print(selected_features)


Selected Features:
                   Feature  Importance
sepal_width    sepal_width    0.540427
petal_length  petal_length    0.766369


In [44]:
print("Number of Selected Features:", len(selected_features))

Number of Selected Features: 2


In [45]:
#Create Reduced Dataset
selected_columns = selected_features['Feature'].tolist()

X_train_selected = X_train[selected_columns]
X_test_selected = X_test[selected_columns]

print("\nFeatures Used:")
print(selected_columns)


Features Used:
['sepal_width', 'petal_length']


#train using random forest model

In [46]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_selected, y_train)

RandomForestClassifier(random_state=42)

In [47]:
y_pred = rf_model.predict(X_test_selected)

In [48]:
accuracy = accuracy_score(y_test, y_pred)


In [51]:
accuracy

0.9

#Using PCA

In [52]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

In [53]:
#Standardize the Features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [54]:
#Apply PCA
#Here we keep 2 principal components.

pca = PCA(n_components=2)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [55]:
#Check Explained Variance
explained_variance = pd.DataFrame({
    'Principal Component': ['PC1', 'PC2'],
    'Explained Variance Ratio': pca.explained_variance_ratio_
})

print(explained_variance)

  Principal Component  Explained Variance Ratio
0                 PC1                  0.726772
1                 PC2                  0.230667


#Train Random Forest

In [56]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_pca, y_train)

RandomForestClassifier(random_state=42)

In [57]:
y_pred = rf_model.predict(X_test_pca)

In [58]:
accuracy= accuracy_score(y_test, y_pred)

In [60]:
accuracy

0.9

#according to the IRIS dataset RFE has given best performance
cause it chose the best features. LASSO selected a different subset

LASSO selected:

sepal_width

petal_length

Although these are useful, petal_width is generally more informative than sepal_width for the Iris dataset. Therefore, Random Forest had slightly less information available, resulting in 90% accuracy.


Random Forest is a tree-based model that often performs better on the original feature space. Transforming the features with PCA can make the tree splits less intuitive, which may reduce performance slightly.


PCA is more commonly helpful for models such as:

Support Vector Machine (SVM)
K-Nearest Neighbors (KNN)
Logistic Regression